# 📝 가설검정·회귀 과제 LV3(통합) — 뉴욕 택시 요금 구조 분석

> 뉴욕시 택시 운행 기록 **6,433건**입니다. 지금까지 과제는 이미 깨끗한 데이터로 시작했지만, **실무 데이터는 그렇지 않습니다.** 이 원본에는 **결측치·잘못된 값·문자로 저장된 날짜**가 섞여 있어 **전처리부터 직접 해야** 분석을 시작할 수 있습니다.

**전처리 → 가정 점검 → 집단 비교 검정 → 회귀 → 의사결정** 을 세 문제로 이어 갑니다. 각 `### N단계` 셀에 그 단계에서 할 일이 자립적으로 적혀 있어요.

## 데이터 설명 (`taxis.csv`)
| 열 | 뜻 |
|---|---|
| `pickup` · `dropoff` | 승·하차 시각 — **문자열로 저장**되어 있어 날짜형으로 바꿔야 합니다 |
| `passengers` | 승객 수 |
| `distance` | 주행 거리(마일) |
| `fare` · `tip` · `tolls` · `total` | 기본요금 · 팁 · 통행료 · 총액(달러) |
| `color` | 택시 종류(`yellow` 노란 택시 / `green` 외곽 전용 택시) |
| `payment` | 결제수단(`credit card` / `cash`) |
| `pickup_zone` · `dropoff_zone` | 승·하차 세부 지역 |
| `pickup_borough` · `dropoff_borough` | 승·하차 자치구(Manhattan · Queens · Brooklyn · Bronx) |

## 풀이 방법
1. **문제 1(전처리)을 먼저 끝내세요.** 마지막 단계에서 정제본을 파일로 저장하고, **문제 2·3 은 그 파일을 다시 읽어** 시작합니다(문제 간 오염 방지).
2. 각 단계의 **답안 셀**(`# 여기에 코드를 작성하세요`)을 채웁니다. 정량 단계는 아래 **자가채점 셀**로 확인하고, **그래프 단계는 자가채점 없이** 위 **완성 그래프(정답)** 와 같은 모양으로 그립니다.
3. **해석·의사결정 단계**는 서술형입니다 — 정답 노트북의 모범 서술과 비교하세요.

화이팅!

> 🔧 **이번 과제의 도구**: 검정은 이번 단원 주력인 **Pingouin**(`pg.normality`·`pg.homoscedasticity`·`pg.mwu`·`pg.kruskal`·`pg.chi2_independence`)으로, **회귀는 `statsmodels`**(`smf.ols`)로 갑니다. 전처리는 6일차에 배운 **pandas**(`to_datetime`·`dropna`·조건 필터·파생 열)를 그대로 씁니다.

In [ ]:
# [제공 코드] 전처리·검정·회귀에 쓸 라이브러리와 한글 폰트를 준비합니다.
import warnings
warnings.filterwarnings('ignore')   # 사소한 경고를 숨겨 출력을 깔끔하게
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
import statsmodels.formula.api as smf

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={"axes.unicode_minus": False})

os.makedirs('output', exist_ok=True)          # 정제본을 저장할 폴더

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을 봅니다. (아래 셀은 실행만 하면 됩니다.)

**여기서 세 가지를 눈으로 확인하세요** — ① `pickup`·`dropoff` 의 자료형이 `object`(문자열)라는 것, ② 결측이 있는 열이 어디인지, ③ `distance`·`passengers` 의 **최솟값이 0** 이라는 것(거리 0마일·승객 0명인 운행은 정상적인 기록이 아닙니다).

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·요약
#   (미리보기 전용 변수 preview 를 씁니다. 문제 풀이용 df 는 문제 1 에서 직접 불러오세요.)
preview = pd.read_csv("data/taxis.csv")
print("행·열 크기:", preview.shape)
print("\n[앞 5행] head()"); display(preview.head())
print("\n[열·자료형·결측] info()"); preview.info()
print("\n[수치형 요약] describe()"); display(preview.describe().round(3))

---
## 1. 원본 데이터 전처리 — 분석할 수 있는 상태로 만들기

**배경**: 원본에는 ① 문자열로 저장된 날짜, ② 다섯 개 열의 결측치, ③ 물리적으로 불가능한 값(거리 0마일·승객 0명·소요시간 0분)이 섞여 있습니다. 이걸 그대로 검정에 넣으면 결과가 왜곡됩니다. **하나의 `df` 를 이어서** 여섯 단계로 정제합니다.

| 단계 | 확인 항목 |
|---|---|
| 1단계 | 원본 `(6433, 14)`, 결측 있는 열 **5개**, 중복 행 **0개** |
| 2단계 | 날짜형 변환 + 파생 열 `소요시간_분` — 중앙값 **10.9** |
| 3단계 | 파생 열 `팁비율` (= `tip` ÷ `fare`) — 평균 **0.1692**, 열 수 **16** |
| 4단계 | 결측 행 제거 후 **6341** 행 |
| 5단계 | 이상치 행 제거 후 **6220** 행 |
| 6단계 | 정제본 저장 — `output/taxis_정제.csv` |

### 1단계 — 불러오기·구조 파악
**요구사항**:
- `data/taxis.csv` 를 `df` 로 불러오세요(**날짜 변환은 아직 하지 않습니다** — 2단계에서 합니다).
- 원본의 **행 수**를 `raw_rows`, **열 수**를 `raw_cols` 에 담으세요.
- **결측치가 하나라도 있는 열의 개수**를 `n_missing_cols` 에 담으세요(정수).
- **완전히 똑같이 중복된 행의 개수**를 `n_dup` 에 담으세요(정수).
- 위 네 값을 `print` 로 출력해 확인하세요.

**예시**
```
raw_rows        → 6433
raw_cols        → 14
n_missing_cols  → 5
n_dup           → 0
```

<details><summary>힌트</summary>

```text
접근방법:
- 결측 개수는 열별로 세어(isna 합계) 그 값이 0 보다 큰 열이 몇 개인지 센다.
- 중복 행은 duplicated 결과의 합이다.

세부구현:
1. read_csv 로 원본을 df 에 담는다
2. shape 에서 행 수와 열 수를 꺼낸다
3. 열별 결측 개수를 구해 0 보다 큰 것의 개수를 int 로 담는다
4. 중복 행 개수를 int 로 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert raw_rows == 6433
assert raw_cols == 14
assert n_missing_cols == 5, '결측이 하나라도 있는 열이 몇 개인지 세세요'
assert n_dup == 0
print("✅ 1단계 통과!")

### 2단계 — 날짜형 변환과 파생 열 `소요시간_분`
**배경**: `pickup`·`dropoff` 는 **문자열**이라 그대로는 뺄셈이 안 됩니다. 날짜형으로 바꾼 뒤 두 시각의 차이로 **운행 소요시간**을 만듭니다. 소요시간은 뒤에서 회귀의 설명변수로 씁니다.

**요구사항**:
- `pd.to_datetime` 으로 `df['pickup']` 과 `df['dropoff']` 를 **날짜형으로 변환**하세요.
- 두 시각의 차이를 **분 단위 실수**로 만들어 `df['소요시간_분']` 열에 담으세요.
  (시간 차이는 `Timedelta` 라서 `.dt.total_seconds()` 로 초를 얻은 뒤 60 으로 나눕니다.)
- `소요시간_분` 의 **중앙값**을 `dur_median`, **최솟값**을 `dur_min` 에 담으세요.
- 중앙값·최솟값·최댓값을 `print` 로 출력하세요. **최솟값이 0** 이라는 점을 눈으로 확인하세요(5단계에서 처리합니다).

**예시**
```
round(dur_median, 2)  → 10.9
round(dur_min, 2)     → 0.0     # 0분짜리 운행 = 비정상 기록
```

<details><summary>힌트</summary>

```text
접근방법:
- 문자열 날짜를 날짜형으로 바꾸는 pandas 함수를 두 열에 각각 적용한다.
- 두 날짜형 열을 빼면 Timedelta 가 나온다. 초로 바꾼 뒤 60 으로 나누면 분이다.

세부구현:
1. pickup·dropoff 를 각각 날짜형으로 변환해 같은 열에 다시 담는다
2. (dropoff - pickup) 의 .dt.total_seconds() 를 60 으로 나눠 소요시간_분 열을 만든다
3. 그 열의 median·min 을 dur_median·dur_min 에 담고 max 와 함께 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert str(df['pickup'].dtype).startswith('datetime'), 'pickup 을 날짜형으로 변환했는지 확인하세요'
assert str(df['dropoff'].dtype).startswith('datetime')
assert '소요시간_분' in df.columns
assert abs(dur_median - 10.9) < 0.01
assert abs(dur_min - 0.0) < 0.01
print("✅ 2단계 통과!")

### 3단계 — 파생 열 `팁비율`
**배경**: 팁 금액(`tip`)을 그대로 비교하면 **요금이 비싼 운행일수록 팁도 큰** 당연한 결과가 나옵니다. '얼마나 후하게 줬는가'를 보려면 **기본요금 대비 비율**로 바꿔야 공정합니다.

**요구사항**:
- `tip` 을 `fare` 로 나눈 값을 `df['팁비율']` 열에 담으세요.
- `팁비율` 의 **평균**을 `tip_rate_mean` 에, 이 시점의 **열 개수**를 `n_cols_after` 에 담으세요.
- 평균·최댓값과 열 개수를 `print` 로 출력하세요.

**예시**
```
round(tip_rate_mean, 4) → 0.1692
n_cols_after            → 16      # 원래 14 + 소요시간_분 + 팁비율
```

<details><summary>힌트</summary>

```text
접근방법:
- 두 열끼리 나누면 행마다 계산된 새 Series 가 나온다. 그것을 새 열에 담는다.

세부구현:
1. tip 열을 fare 열로 나눠 팁비율 열을 만든다
2. 팁비율의 mean 을 tip_rate_mean 에 담는다
3. df.shape 의 열 개수를 n_cols_after 에 담고 함께 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert '팁비율' in df.columns
assert abs(tip_rate_mean - 0.1692) < 0.001, '팁비율 = tip ÷ fare 입니다'
assert n_cols_after == 16
print("✅ 3단계 통과!")

### 4단계 — 결측 행 제거
**배경**: 결측 행을 버릴 때는 **어느 열을 기준으로 볼지 먼저 정하는 것**이 원칙입니다. 이번 분석에 실제로 쓸 열은 `payment`·`pickup_borough`·`dropoff_borough` 세 개이므로, 그 세 열만 기준으로 삼습니다.

**요구사항**:
- `payment`·`pickup_borough`·`dropoff_borough` **세 열 기준으로만** 결측 행을 제거해 `df` 에 다시 담으세요.
- 제거 후 남은 행 수를 `rows_after_na`, **제거된 행 수**를 `removed_na` 에 담으세요.
- 두 값을 `print` 로 출력하세요.

**예시**
```
rows_after_na → 6341
removed_na    → 92
```

<details><summary>힌트</summary>

```text
접근방법:
- 결측 행 제거 함수에 '어느 열을 기준으로 볼지' 를 지정하는 인자가 있다. 세 열만 넘긴다.
- 제거된 행 수는 (제거 전 행 수 − 제거 후 행 수) 로 구한다.

세부구현:
1. 제거 전 행 수를 미리 변수에 담아 둔다
2. payment·pickup_borough·dropoff_borough 세 열을 기준으로 결측 행을 없애 df 에 다시 담는다
3. 남은 행 수와 (전 − 후) 를 각각 rows_after_na, removed_na 에 담아 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert rows_after_na == 6341, 'payment·pickup_borough·dropoff_borough 세 열만 기준으로 제거하세요'
assert removed_na == 92
assert df['payment'].isna().sum() == 0
print("✅ 4단계 통과!")

### 5단계 — 이상치(불가능한 값) 제거
**배경**: 거리 0마일, 소요시간 0분, 승객 0명인 운행은 **현실에서 있을 수 없는 기록**입니다. 미터기 오작동이나 취소된 호출로 보이며, 그대로 두면 회귀의 기울기를 끌어당깁니다.

**요구사항**:
- `distance > 0` **그리고** `소요시간_분 > 0` **그리고** `passengers > 0` 인 행만 남겨 `df` 에 다시 담으세요.
- 남은 행 수를 `rows_clean`, **제거된 행 수**를 `removed_outlier` 에 담으세요.
- 두 값과 함께, 정제 후 `total`(총액)의 **평균**을 `print` 로 출력하세요.

**예시**
```
rows_clean      → 6220
removed_outlier → 121
```

<details><summary>힌트</summary>

```text
접근방법:
- 세 조건을 모두 만족하는 행만 남긴다. pandas 에서 조건 여러 개는 & 로 잇고 각 조건을 괄호로 감싼다.

세부구현:
1. 제거 전 행 수를 미리 담아 둔다
2. 세 조건(거리·소요시간·승객이 모두 0보다 큼)을 & 로 이어 걸러 df 에 다시 담는다
3. 남은 행 수와 제거된 행 수를 각각 rows_clean, removed_outlier 에 담아 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert rows_clean == 6220, '세 조건(거리·소요시간·승객 > 0)을 모두 걸었는지 확인하세요'
assert removed_outlier == 121
assert (df['distance'] > 0).all() and (df['passengers'] > 0).all()
print("✅ 5단계 통과!")

### 6단계 — 정제본 저장
**배경**: 전처리 결과를 파일로 남겨 두면 **뒤 분석은 정제본만 읽어** 바로 시작할 수 있습니다. 실무에서도 '원본은 절대 덮어쓰지 않고, 정제본을 따로 저장' 하는 것이 기본입니다.

**요구사항**:
- 정제된 `df` 를 `output/taxis_정제.csv` 로 저장하세요(**인덱스는 저장하지 않습니다** — `index=False`).
- 저장한 파일을 다시 읽어 `check` 에 담고, 그 **행 수**를 `saved_rows`, **열 수**를 `saved_cols` 에 담으세요.
- 최종 크기를 `print` 로 출력하세요.

**예시**
```
saved_rows → 6220
saved_cols → 16
```

<details><summary>힌트</summary>

```text
접근방법:
- DataFrame 을 CSV 로 내보내는 메서드에 index=False 를 준다.
- 저장이 제대로 됐는지 확인하려면 그 파일을 다시 읽어 크기를 본다.

세부구현:
1. df 를 output/taxis_정제.csv 로 저장한다(index=False)
2. 같은 경로를 read_csv 로 다시 읽어 check 에 담는다
3. check.shape 에서 행 수·열 수를 꺼내 saved_rows, saved_cols 에 담고 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert saved_rows == 6220
assert saved_cols == 16
assert os.path.exists('output/taxis_정제.csv'), '정제본을 저장했는지 확인하세요'
print("✅ 6단계 통과! — 전처리 완료, 이제 분석을 시작할 수 있습니다")

---
## 2. 어떤 집단이 다른가 — 가정 점검과 비교 검정

**배경**: 정제본으로 세 가지 질문에 답합니다. **① 결제수단에 따라 팁을 더 후하게 주는가?** **② 승차 자치구에 따라 팁비율이 다른가?** **③ 택시 종류와 자치구는 서로 관련이 있는가?** 검정을 고르기 전에 **가정부터 점검**하고, 결과가 나오면 **그 숫자가 어떻게 만들어졌는지** 확인합니다.

> ⚠️ **문제 1 을 먼저 끝내세요.** 이 문제는 문제 1 이 저장한 정제본을 **새로 읽어** 시작합니다.

| 단계 | 확인 항목 |
|---|---|
| 1단계 | 정제본 로드 — **6220** 행, 결제수단별 건수 |
| 2단계 | 정규성(카드 팁비율) · 등분산 점검 → 검정 선택 |
| 3단계 | 결제수단별 팁비율 — Mann-Whitney U **7469831.0**, CLES **0.9506** |
| 4단계 | **결과 뒤집어 보기** — 현금 결제의 팁비율 0 비율 **1.0** |
| 5단계 | 카드 결제만으로 자치구별 팁비율 — Kruskal H **544.18** |
| 6단계 | 택시 종류 × 승차 자치구 카이제곱 — Cramér's V **0.621** |

### 1단계 — 정제본 불러오기
**요구사항**:
- 문제 1 에서 저장한 `output/taxis_정제.csv` 를 `taxi` 에 불러오세요(**문제 1 의 `df` 를 이어 쓰지 말고 새로 읽습니다** — 문제 간 오염 방지).
- 행 수를 `n_clean` 에 담으세요.
- 결제수단(`payment`)별 건수를 `pay_counts`(Series)에 담고 출력하세요.

**예시**
```
n_clean                    → 6220
pay_counts['credit card']  → 4457
pay_counts['cash']         → 1763
```

<details><summary>힌트</summary>

```text
접근방법:
- 정제본 경로를 read_csv 로 읽는다.
- 범주별 건수는 그 열의 값 세기 메서드로 구한다.

세부구현:
1. output/taxis_정제.csv 를 taxi 에 담는다
2. taxi.shape 의 행 수를 n_clean 에 담는다
3. payment 열의 값별 개수를 pay_counts 에 담아 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_clean == 6220, '문제 1 의 정제본(6220행)을 새로 읽었는지 확인하세요'
assert pay_counts['credit card'] == 4457
assert pay_counts['cash'] == 1763
print("✅ 1단계 통과!")

### 2단계 — 가정 점검: 정규성과 등분산
**배경**: 두 집단의 팁비율을 비교하기 전에, **평균을 쓰는 t-검정을 써도 되는지** 확인합니다. 정규성은 `pg.normality`, 등분산은 `pg.homoscedasticity`(Levene)로 봅니다.

**요구사항**:
- **카드 결제(`payment == 'credit card'`)** 행의 `팁비율` 을 `rate_card` 에 담으세요.
- `pg.normality` 로 `rate_card` 의 정규성을 검정해 **W**(`['W'].iloc[0]`)를 `norm_w`, **pval**(`['pval'].iloc[0]`)을 `norm_p` 에 담으세요.
- `pg.homoscedasticity(data=taxi, dv='팁비율', group='payment')` 로 등분산을 검정해 **pval**(`['pval'].iloc[0]`)을 `levene_p` 에 담으세요.
- 세 값을 출력하고, **정규성·등분산이 모두 깨졌다는 것**을 확인하세요.
- 검정통계량은 소수 **셋째 자리**까지 비교합니다.

> 💡 **왜 카드만 검정하나요?** 현금 결제 쪽은 값이 전부 똑같아서 정규성 검정이 아예 계산되지 않습니다(`W` 가 `nan`). **그 자체가 데이터에 뭔가 있다는 신호**인데, 무엇인지는 4단계에서 밝힙니다.

**예시**
```
round(norm_w, 3)  → 0.939
norm_p            → 1.6e-39   (0.05 보다 훨씬 작음 → 정규성 기각)
levene_p          → 0.0       (0.05 보다 작음 → 등분산도 기각)
```

<details><summary>힌트</summary>

```text
접근방법:
- 결제수단으로 행을 걸러 팁비율 열만 꺼낸다.
- 정규성은 그 시리즈에, 등분산은 긴 형태(dv=팁비율, group=payment)로 점검한다.

세부구현:
1. payment 가 'credit card' 인 행의 팁비율을 rate_card 에 담는다
2. pg.normality(rate_card) 결과의 ['W']·['pval'] 첫 값을 norm_w, norm_p 에 담는다
3. pg.homoscedasticity 로 등분산 pval 을 levene_p 에 담는다
4. 세 값을 출력하고 0.05 와 비교해 해석한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(norm_w - 0.939) < 0.01
assert norm_p < 0.05, '정규성이 기각되는 것이 정상입니다'
assert levene_p < 0.05
print("✅ 2단계 통과!")

### 3단계 — 결제수단별 팁비율 비교 (Mann-Whitney U)
**배경**: 가정이 깨졌으니 **비모수 검정**으로 '카드 결제와 현금 결제의 팁비율이 다른가'를 봅니다.

**요구사항**:
- 현금 결제(`payment == 'cash'`) 행의 `팁비율` 을 `rate_cash` 에 담으세요.
- `pg.mwu(rate_card, rate_cash)` 로 검정해 결과 표를 `display` 하세요.
- 결과 표에서 **U_val** 을 `u_stat`, **p_val** 을 `mwu_p`, 효과크기 **CLES** 를 `cles` 에 담으세요.
- 세 값을 출력하세요. `cles` 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
u_stat          → 7469831.0
mwu_p           → 0.0        (사실상 0)
round(cles, 3)  → 0.951      # 카드 쪽이 더 클 확률이 95%
```
> **CLES(공통언어 효과크기)** 는 '앞 집단에서 하나, 뒤 집단에서 하나를 뽑았을 때 앞이 더 클 확률'입니다. 0.5 면 차이 없음, 1.0 에 가까울수록 앞 집단이 압도적으로 큽니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 독립 집단의 비모수 비교 함수에 두 시리즈를 순서대로 넘긴다(카드를 앞에).
- 결과 표에서 검정통계량·p-value·CLES 열을 꺼낸다.

세부구현:
1. payment 가 'cash' 인 행의 팁비율을 rate_cash 에 담는다
2. 비모수 2집단 검정 함수를 (rate_card, rate_cash) 순서로 호출해 결과 표를 받아 display 한다
3. 결과 표의 U_val·p_val·CLES 첫 값을 u_stat, mwu_p, cles 에 담아 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(u_stat - 7469831.0) < 1, '카드(rate_card)를 첫 인자로 넣었는지 확인하세요'
assert mwu_p < 0.001
assert abs(cles - 0.951) < 0.01
print("✅ 3단계 통과!")

### 4단계 — 결과를 뒤집어 보기: 이 차이는 진짜인가
**배경**: p 는 사실상 0, CLES 는 0.95 로 **엄청난 차이**가 나왔습니다. 여기서 "현금 손님은 팁에 인색하다" 고 결론 내리면 **틀립니다.** 결론을 내기 전에 **그 숫자가 어떻게 만들어졌는지** 반드시 확인하세요.

**요구사항**:
- 결제수단별 `팁비율` 의 **평균·중앙값**을 구해 출력하세요.
- 현금 결제 중 **팁비율이 정확히 0인 행의 비율**을 `cash_zero_ratio` 에, 카드 결제 중 같은 비율을 `card_zero_ratio` 에 담으세요.
- 두 비율을 출력하고, **무엇이 이상한지** 눈으로 확인하세요.

**예시**
```
cash_zero_ratio → 1.0      # 현금 결제는 100% 가 팁 0
round(card_zero_ratio, 3) → 0.099
```

<details><summary>힌트</summary>

```text
접근방법:
- 결제수단으로 묶어 팁비율의 평균·중앙값을 함께 구한다.
- '값이 0인 비율' 은 (시리즈 == 0) 의 평균으로 구할 수 있다(True=1, False=0).

세부구현:
1. payment 로 groupby 해 팁비율의 mean·median 을 구해 출력한다
2. rate_cash 에서 (값 == 0) 의 평균을 cash_zero_ratio 에 담는다
3. rate_card 에서 같은 값을 card_zero_ratio 에 담아 함께 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(cash_zero_ratio - 1.0) < 1e-9, '현금 결제의 팁비율이 0 인 비율입니다 — 정확히 1.0 이 나옵니다'
assert abs(card_zero_ratio - 0.099) < 0.01
print("✅ 4단계 통과! — 검정 결과를 해석하기 전에 데이터가 어떻게 기록됐는지 확인해야 합니다")

### 5단계 — 카드 결제만으로 자치구별 팁비율 비교 (Kruskal-Wallis + 그래프)
**배경**: 4단계에서 현금 데이터를 믿을 수 없다는 것을 알았으니, **카드 결제만 남겨** '승차 자치구에 따라 팁비율이 다른가'를 봅니다. 자치구는 4개(3집단 이상)이고 정규성이 깨졌으므로 **Kruskal-Wallis**(`pg.kruskal`)를 씁니다.

**요구사항**:
- 카드 결제 행만 남긴 데이터프레임을 `card` 에 담으세요.
- `pg.kruskal(data=card, dv='팁비율', between='pickup_borough')` 로 검정해 결과 표를 `display` 하세요.
- 결과 표에서 **H** 를 `h_stat`, **p_unc** 를 `kruskal_p` 에 담으세요.
- 자치구별 팁비율 **중앙값**을 `borough_median`(Series)에 담고 출력하세요.
- `h_stat` 은 소수 **둘째 자리**까지 비교하고, `kruskal_p` 는 0 에 아주 가까워 `< 1e-100` 인지로 채점됩니다.
- 마지막으로 **자치구별 팁비율 상자그림**(`sns.boxplot`)을 그리세요.

**예시**
```
round(h_stat, 2)               → 544.18
borough_median['Manhattan']    → 0.266
borough_median['Bronx']        → 0.0      # 카드인데도 중앙값이 0
```
> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv3_q2_s5.png" width="620"/>

<details><summary>힌트</summary>

```text
접근방법:
- 결제수단이 카드인 행만 남긴다.
- 3집단 이상의 비모수 비교 함수에 data·dv·between 을 이름으로 넘긴다.
- 상자그림은 x 에 자치구, y 에 팁비율을 준다.

세부구현:
1. payment 가 'credit card' 인 행만 남겨 card 에 담는다
2. 비모수 3집단 검정 함수를 data=card, dv='팁비율', between='pickup_borough' 로 호출해 표를 받는다
3. 그 표의 H·p_unc 첫 값을 h_stat, kruskal_p 에 담는다
4. pickup_borough 로 묶어 팁비율의 중앙값을 borough_median 에 담아 출력한다
5. 새 figure 를 열고 상자그림을 그린 뒤 제목·축 라벨을 달고 보여 준다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(h_stat - 544.18) < 0.05, '카드 결제만 남긴 뒤 검정했는지 확인하세요'
assert kruskal_p < 1e-100
assert abs(borough_median['Manhattan'] - 0.266) < 0.01
assert abs(borough_median['Bronx'] - 0.0) < 0.01
print("✅ 5단계 통과!")

### 6단계 — 택시 종류와 자치구는 관련이 있는가 (카이제곱)
**배경**: 5단계 끝의 의문 — 'Bronx·Brooklyn 은 다른 종류의 택시가 다니는 것 아닐까?' 를 확인합니다. **택시 종류(`color`)와 승차 자치구(`pickup_borough`)** 는 둘 다 범주형이므로 **카이제곱 독립성 검정**으로 관련성을 보고, 세기는 **Cramér's V** 로 잽니다.

**요구사항**:
- `pg.chi2_independence(data=taxi, x='color', y='pickup_borough')` 를 실행하세요. 이 함수는 **(기대빈도표, 관측빈도표, 통계량표)** 세 개를 순서대로 돌려줍니다. 관측 교차표를 `display` 하세요.
- 통계량표에서 **표준 Pearson 검정 행**(`test == 'pearson'`)을 골라 `chi2`(→`chi2`), `pval`(→`chi_p`), `cramer`(→`cramers_v`) 를 꺼내세요.
- 세 값을 출력하세요. 검정통계량·Cramér's V 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
round(chi2, 3)      → 2401.338
chi_p               → 0.0
round(cramers_v, 3) → 0.621     # 0.5 이상이면 매우 강한 연관
```

<details><summary>힌트</summary>

```text
접근방법:
- 두 범주형 변수의 관련성은 카이제곱 독립성 검정으로 본다 — (기대, 관측, 통계량) 세 표를 준다.
- 통계량표의 test=='pearson' 행이 표준 카이제곱이고, cramer 열이 효과크기다.

세부구현:
1. expected, observed, chi_stats = pg.chi2_independence(data=taxi, x='color', y='pickup_borough')
2. observed 를 display 해 교차표를 눈으로 본다
3. chi_stats 에서 test 가 'pearson' 인 행을 골라 chi2·pval·cramer 의 첫 값을 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(chi2 - 2401.338) < 0.01
assert chi_p < 0.001
assert abs(cramers_v - 0.621) < 0.01, "통계량표 pearson 행의 cramer 열이 Cramér's V 예요"
print("✅ 6단계 통과!")

**해석 (서술)**

*(여기에 3문장 이상으로 서술하세요 — ① 3단계의 '결제수단별 팁비율 차이'를 왜 그대로 결론으로 쓸 수 없는지, ② 5단계에서 카드 결제만 남긴 이유, ③ 6단계 결과를 볼 때 '자치구가 팁비율의 원인'이라고 말할 수 있는지)*

---
## 3. 요금은 무엇으로 정해지는가 — 회귀와 의사결정

**배경**: 마지막 질문입니다. **택시 총액(`total`)은 무엇으로 설명되는가?** 거리 하나로 시작해 소요시간·승객 수를 더해 가며, 각 변수의 기여를 계수와 p-value 로 읽고 **이 회귀를 믿어도 되는지** 진단까지 합니다.

> ⚠️ 이 문제도 정제본을 **새로 읽어** 시작합니다.

| 단계 | 확인 항목 |
|---|---|
| 1단계 | 정제본 로드 — **6220** 행 |
| 2단계 | 단순회귀 `total ~ distance` — 기울기 **3.242**, R² **0.882** |
| 3단계 | 다중회귀 `+ 소요시간_분 + passengers` — Adj R² **0.906** |
| 4단계 | VIF — `distance` **3.094**, `소요시간_분` **3.094** |
| 5단계 | 잔차 진단 그래프 — Durbin-Watson **1.781** |
| 6단계 | 예측과 의사결정 서술 |

### 1단계 — 정제본 다시 불러오기
**요구사항**:
- `output/taxis_정제.csv` 를 `tx` 에 불러오세요(문제 2 의 `taxi` 를 이어 쓰지 말고 **새로 읽습니다**).
- 행 수를 `n_reg` 에 담고, `total`(총액)의 **평균**을 `total_mean` 에 담아 출력하세요.
- `total_mean` 은 소수 **셋째 자리**까지 비교합니다.

**예시**
```
n_reg                  → 6220
round(total_mean, 3)   → 18.303
```

<details><summary>힌트</summary>

```text
접근방법:
- 문제 1 이 저장한 정제본을 read_csv 로 읽는다.

세부구현:
1. output/taxis_정제.csv 를 tx 에 담는다
2. 행 수를 n_reg, total 열의 평균을 total_mean 에 담아 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_reg == 6220
assert abs(total_mean - 18.303) < 0.01
print("✅ 1단계 통과!")

### 2단계 — 단순 선형회귀: 거리로 총액 설명하기
**배경**: 가장 단순한 모형부터 시작합니다. **거리 하나로 총액이 얼마나 설명되는가?**

**요구사항**:
- `smf.ols('total ~ distance', data=tx).fit()` 로 적합해 `m1` 에 담으세요.
- 절편을 `b0`, `distance` 계수를 `b_dist`, 결정계수를 `r2_simple` 에 담으세요.
- 세 값을 출력하고, **거리가 1마일 늘 때 총액이 얼마나 오르는지** 문장으로 함께 출력하세요.
- 계수·R² 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
round(b0, 3)        → 8.531
round(b_dist, 3)    → 3.242    # 1마일당 약 3.24달러
round(r2_simple, 3) → 0.882
```

<details><summary>힌트</summary>

```text
접근방법:
- ols 에 '종속변수 ~ 독립변수' 식과 데이터를 넣고 fit 한다.
- 적합 결과 객체에서 params 로 계수를, rsquared 로 결정계수를 꺼낸다.

세부구현:
1. total 을 distance 로 설명하는 식으로 모형을 적합해 m1 에 담는다
2. params['Intercept']·params['distance'] 를 b0, b_dist 에 담는다
3. rsquared 를 r2_simple 에 담고 세 값을 해석 문장과 함께 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(b0 - 8.531) < 0.01
assert abs(b_dist - 3.242) < 0.01
assert abs(r2_simple - 0.882) < 0.01
print("✅ 2단계 통과!")

### 3단계 — 다중 선형회귀: 소요시간과 승객 수를 더하면
**배경**: 택시 요금은 거리뿐 아니라 **막힌 시간**에도 붙습니다(시간 요금). 소요시간과 승객 수를 넣어 설명력이 얼마나 오르는지, 각 변수가 유의한지 봅니다.

**요구사항**:
- `smf.ols('total ~ distance + 소요시간_분 + passengers', data=tx).fit()` 로 적합해 `m2` 에 담으세요.
- `distance` 계수를 `c_dist`, `소요시간_분` 계수를 `c_dur`, `passengers` 계수를 `c_pass` 에 담으세요.
- `passengers` 계수의 p-value 를 `p_pass`, 모형의 **조정 결정계수**를 `adj_r2` 에 담으세요.
- `print(m2.summary())` 로 전체 표를 보고, 위 값들을 함께 출력하세요.
- 계수·조정 R² 는 소수 **셋째 자리**, p값은 소수 **넷째 자리**까지 비교합니다.

**예시**
```
round(c_dist, 3)   → 2.476     # 소요시간을 넣자 3.242 에서 줄었다
round(c_dur, 3)    → 0.302
round(c_pass, 3)   → 0.100
round(p_pass, 4)   → 0.0159    # 유의하긴 하다
round(adj_r2, 3)   → 0.906
```

<details><summary>힌트</summary>

```text
접근방법:
- ols 식에 독립변수를 + 로 이어 세 개를 넣는다.
- 조정 결정계수는 rsquared_adj 로 꺼낸다.

세부구현:
1. total 을 distance·소요시간_분·passengers 로 설명하는 식으로 적합해 m2 에 담는다
2. params 에서 세 계수를, pvalues 에서 passengers 의 p 를 꺼낸다
3. rsquared_adj 를 adj_r2 에 담고 summary 와 함께 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(c_dist - 2.476) < 0.01
assert abs(c_dur - 0.302) < 0.01
assert abs(c_pass - 0.100) < 0.01
assert abs(p_pass - 0.0159) < 0.001
assert abs(adj_r2 - 0.906) < 0.01
print("✅ 3단계 통과!")

### 4단계 — 다중공선성 점검 (VIF)
**배경**: '거리'와 '소요시간'은 서로 강하게 얽혀 있을 것 같습니다(멀면 오래 걸리죠). 얼마나 얽혔는지 **VIF**(분산팽창인자)로 재 봅니다. 한 변수를 **나머지 변수들로 회귀**했을 때의 결정계수 $R_j^2$ 로 $VIF_j = 1/(1-R_j^2)$ 입니다.

**요구사항**:
- 세 독립변수 각각에 대해 **나머지 두 변수로 회귀**한 보조 모형의 R² 를 구하고, `1/(1-R²)` 로 VIF 를 계산하세요.
- 결과를 `{'변수': ..., 'VIF': ...}` 형태의 `vif_table`(DataFrame)로 만들어 `display` 하세요.
- `distance` 의 VIF 를 `vif_dist`, `passengers` 의 VIF 를 `vif_pass` 에 담으세요.
- VIF 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
round(vif_dist, 3) → 3.094    # 5 미만이라 해석 가능한 수준
round(vif_pass, 3) → 1.000    # 다른 변수와 사실상 무관
```

<details><summary>힌트</summary>

```text
접근방법:
- 변수 목록을 for 로 돌면서, 그 변수를 종속변수로 두고 나머지를 독립변수로 하는 보조 회귀를 적합한다.
- 보조 회귀의 결정계수로 1/(1-R^2) 을 계산한다.

세부구현:
1. 독립변수 이름 세 개를 리스트에 담는다
2. 각 이름마다 나머지 이름들을 ' + ' 로 이어 식 문자열을 만들어 ols 로 적합한다
3. 그 모형의 rsquared 로 VIF 를 계산해 리스트에 모으고 DataFrame 으로 만든다
4. 표에서 distance·passengers 의 VIF 를 꺼내 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(vif_dist - 3.094) < 0.01
assert abs(vif_pass - 1.000) < 0.01
assert len(vif_table) == 3
print("✅ 4단계 통과!")

### 5단계 — 잔차 진단: 이 회귀를 믿어도 되는가 (그래프)
**배경**: 회귀가 성립하려면 잔차(실제값 − 예측값)가 **무늬 없이 0 주변에 고르게** 흩어지고(등분산·선형성), 대체로 **정규분포**를 따라야 합니다. 두 그림으로 눈으로 진단하고, **독립성**은 Durbin-Watson 으로 확인합니다.

**요구사항**:
- `plt.subplots(1, 2, figsize=(12, 5))` 로 서브플롯 두 개(`ax1`·`ax2`)를 만드세요.
- **왼쪽(`ax1`)**: x=적합값(`m2.fittedvalues`), y=잔차(`m2.resid`)의 **산점도**를 그리고(점이 많으니 `alpha=0.3`), 잔차 0 위치에 빨간 **수평 점선**을 그으세요.
- **오른쪽(`ax2`)**: 잔차의 **Q-Q Plot** 을 `pg.qqplot(m2.resid, dist='norm', ax=ax2)` 로 그리세요.
- 두 Axes 에 각각 제목을 다세요.
- **Durbin-Watson** 을 정의대로 직접 계산해 `dw` 에 담고 출력하세요(`rv = m2.resid.to_numpy()` 로 꺼낸 뒤 `np.sum(np.diff(rv)**2) / np.sum(rv**2)`).
- `dw` 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
round(dw, 3) → 1.781    # 2 근처면 잔차가 서로 독립
```
> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv3_q3_s5.png" width="760"/>

<details><summary>힌트</summary>

```text
접근방법:
- 서브플롯 두 개를 나란히 만들어 왼쪽엔 적합값-잔차 산점도, 오른쪽엔 잔차 Q-Q Plot 을 그린다.
- Durbin-Watson 은 이웃한 잔차의 차이 제곱합을 잔차 제곱합으로 나눈 값이다.

세부구현:
1. subplots(1, 2, ...) 로 ax1, ax2 두 축을 만든다
2. ax1 에 fittedvalues(x)-resid(y) 산점도를 alpha 를 낮춰 그리고 axhline 으로 0 선을 긋는다
3. ax2 에 pg.qqplot(잔차, dist='norm', ax=ax2) 로 Q-Q Plot 을 그린다
4. 두 축에 제목을 달고 보여 준다
5. 잔차를 numpy 배열로 꺼내 DW 를 직접 계산해 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(dw - 1.781) < 0.01
print("✅ 5단계 통과!")

### 6단계 — 예측과 의사결정
**배경**: 마지막으로 모형을 **실제 예측**에 써 보고, 지금까지의 분석을 하나의 결론으로 정리합니다.

**요구사항**:
- 3단계의 `m2` 로 **거리 3.0마일 · 소요시간 15분 · 승객 1명** 인 운행의 총액을 예측해 `pred_total` 에 담으세요.
  (`m2.predict(pd.DataFrame({'distance': [3.0], '소요시간_분': [15.0], 'passengers': [1]}))` 의 첫 값)
- 예측값을 출력하세요. 소수 **셋째 자리**까지 비교합니다.
- 아래 **서술 셀**에 의사결정 리포트를 적으세요.

**예시**
```
round(pred_total, 3) → 18.393
```

<details><summary>힌트</summary>

```text
접근방법:
- 적합된 모형의 predict 에 예측하려는 조건을 담은 DataFrame 을 넘긴다(열 이름은 학습 때와 같아야 한다).

세부구현:
1. distance·소요시간_분·passengers 를 각각 한 값씩 담은 DataFrame 을 만든다
2. m2.predict 에 넘겨 나온 결과의 첫 값을 pred_total 에 담아 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(pred_total - 18.393) < 0.01
print("✅ 6단계 통과! — LV3 완주")

**의사결정 리포트 (서술)**

*(여기에 4문장 이상으로 서술하세요 — ① 총액을 설명하는 데 어떤 변수가 얼마나 기여하는지(계수·단위·조정 R²), ② `passengers` 처럼 '유의하지만 실질적으로 작은' 변수를 어떻게 다뤄야 하는지, ③ 잔차 진단에서 발견한 한계, ④ 이 분석 결과를 실제 의사결정에 쓸 때의 주의점)*